In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install scikit-image opencv-python --quiet

import os
import cv2
import numpy as np
from skimage.feature import hog
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report, accuracy_score

In [3]:
def extract_features(img_path, img_size=(128,128)):
    # đọc ảnh và resize
    img = cv2.imread(img_path)
    img = cv2.resize(img, img_size)

    # 3.1 HOG trên ảnh xám
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    hog_feat = hog(
        gray,
        orientations=9,
        pixels_per_cell=(8,8),
        cells_per_block=(2,2),
        block_norm='L2-Hys',
        visualize=False,
        feature_vector=True
    )

    # 3.2 Histogram màu HSV
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    hist = cv2.calcHist(
        [hsv],
        channels=[0,1,2],
        mask=None,
        histSize=[8,8,8],
        ranges=[0,180,0,256,0,256]
    ).flatten()

    # 3.3 Nối và chuẩn hóa L2
    feat = np.hstack([hog_feat, hist])
    norm = np.linalg.norm(feat)
    return feat / norm if norm > 0 else feat

In [15]:
# 4. Load data và trích feature
data_dir = '/content/drive/MyDrive/Data'
labels = []
features = []

for label, sub in enumerate(['Cat','Dog']):  # 0 = Cat, 1 = Dog
    folder = os.path.join(data_dir, sub)
    for fname in os.listdir(folder):
        path = os.path.join(folder, fname)
        try:
            feat = extract_features(path)
            features.append(feat)
            labels.append(label)
        except:
            continue

X = np.array(features)
y = np.array(labels)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [26]:
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('pca',    PCA(n_components=0.95, svd_solver='full')),
    ('svm',    SVC(
        kernel='poly',
        C=1,
        degree=2,
        coef0=1.0
    ))
])

pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.723404255319149


In [22]:
# pipe = Pipeline([
#     ('scaler', StandardScaler()),
#     ('pca', PCA(n_components=0.95, svd_solver='full')),  # giữ 95% phương sai
#     ('svm', SVC(kernel='poly'))
# ])

# # grid search để tối ưu C, degree và coef0
# param_grid = {
#     'svm__C':     [1],
#     'svm__degree':[2],
#     'svm__gamma': [1e-4],
#     'svm__coef0': [1.0]
# }

# grid_poly = GridSearchCV(pipe, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
# grid_poly.fit(X_train, y_train)

# print("Best params (poly):", grid_poly.best_params_)
# print("Train accuracy:", grid_poly.best_score_)
# print("Test accuracy:", grid_poly.score(X_test, y_test))


Best params (poly): {'svm__C': 0.5, 'svm__coef0': 1.0, 'svm__degree': 2, 'svm__gamma': 0.0001}
Train accuracy: 0.7193252929798613
Test accuracy: 0.7106382978723405
